# Evaluación de calidad de datos — Online Retail II

Este notebook evalúa **completitud, unicidad, validez, consistencia y plausibilidad** de las dos hojas de `online_retail_II.xlsx`. Examina faltantes en `Description` y `Customer ID`, facturas canceladas, cantidades y precios atípicos, duplicados y coherencia de los registros.

La unidad de análisis es la línea de factura. Las observaciones señaladas son **alertas para revisión**, no evidencia automática de error. El análisis conserva todas las filas; no elimina ni modifica registros de origen.

**Ejecución:** selecciona un kernel de Python que tenga `pandas`, `numpy`, `openpyxl` e `IPython` instalados y ejecuta las celdas en orden (Run All). La integración carga aproximadamente 1,07 millones de filas; cerrar otros notebooks ayuda si el equipo tiene poca memoria.

Si falta alguna dependencia, desde la raiz del proyecto ejecuta:

```powershell
.\.venv-1\Scripts\python.exe -m pip install -r 03_ETL_DATA_QUALITY\requirements.txt
```

Luego reinicia el kernel y selecciona `.venv-1` antes de usar **Run All**.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

EXPECTED_SHEETS = ("Year 2009-2010", "Year 2010-2011")
EXPECTED_COLUMNS = (
    "Invoice", "StockCode", "Description", "Quantity",
    "InvoiceDate", "Price", "Customer ID", "Country",
)
PERIOD_BY_SHEET = {
    "Year 2009-2010": "2009-2010",
    "Year 2010-2011": "2010-2011",
}

current = Path.cwd().resolve()
PROJECT_ROOT = next(
    (path for path in (current, *current.parents)
     if (path / "02_DATOS" / "online_retail_II.xlsx").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "No se encontró 02_DATOS/online_retail_II.xlsx desde el directorio actual ni sus padres."
    )
SOURCE_FILE = PROJECT_ROOT / "02_DATOS" / "online_retail_II.xlsx"
print(f"Fuente: {SOURCE_FILE}")
print(f"Directorio de trabajo: {Path.cwd()}")

Fuente: C:\Users\USER\Documents\PROYECTO-FINAL-MINERIA\02_DATOS\online_retail_II.xlsx
Directorio de trabajo: C:\Users\USER\Documents\PROYECTO-FINAL-MINERIA


## 1. Carga y verificación estructural

Se lee cada hoja de forma explícita con pandas. Se valida el nombre y orden de las hojas y que ambas tengan el esquema esperado. Las dimensiones de DataFrame excluyen la fila de encabezado; también se informa la dimensión de la hoja incluyéndola.

In [2]:
excel = pd.ExcelFile(SOURCE_FILE, engine="openpyxl")
if tuple(excel.sheet_names) != EXPECTED_SHEETS:
    raise ValueError(
        f"Hojas inesperadas: {tuple(excel.sheet_names)}; se esperaban {EXPECTED_SHEETS}"
    )

sheet_frames = {}
sheet_checks = []
for sheet_name in excel.sheet_names:
    frame = pd.read_excel(excel, sheet_name=sheet_name)
    if tuple(frame.columns) != EXPECTED_COLUMNS:
        raise ValueError(f"Esquema inesperado en {sheet_name}: {tuple(frame.columns)}")
    sheet_frames[sheet_name] = frame
    sheet_checks.append({
        "Hoja": sheet_name,
        "Dimensión de datos (filas, columnas)": frame.shape,
        "Dimensión Excel incl. encabezado": (len(frame) + 1, len(frame.columns)),
        "Rango de fechas": f"{frame['InvoiceDate'].min()} — {frame['InvoiceDate'].max()}",
    })
    print()
    print(f"Tipos de datos observados: {sheet_name}")
    display(frame.dtypes.rename("Tipo").to_frame())
    print(f"Muestra — {sheet_name}")
    display(frame.head(3))

display(pd.DataFrame(sheet_checks))


Tipos de datos observados: Year 2009-2010


,Tipo
Invoice,object
StockCode,object
Description,object
Quantity,int64
InvoiceDate,datetime64[us]
Price,float64
Customer ID,float64
Country,str


Muestra — Year 2009-2010


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom



Tipos de datos observados: Year 2010-2011


,Tipo
Invoice,object
StockCode,object
Description,object
Quantity,int64
InvoiceDate,datetime64[us]
Price,float64
Customer ID,float64
Country,str


Muestra — Year 2010-2011


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom


,Hoja,"Dimensión de datos (filas, columnas)",Dimensión Excel incl. encabezado,Rango de fechas
0,Year 2009-2010,"(525461, 8)","(525462, 8)",2009-12-01 07:45:00 — 2010-12-09 20:01:00
1,Year 2010-2011,"(541910, 8)","(541911, 8)",2010-12-01 08:26:00 — 2011-12-09 12:50:00


## 2. Integración y procedencia

Se apilan las hojas sin descartar registros. `Period` conserva la etiqueta del periodo indicada por la hoja y `SourceSheet` registra el nombre original. Estas etiquetas identifican la fuente; no se usan como una regla de fecha de calendario.

In [3]:
frames = []
for sheet_name, frame in sheet_frames.items():
    frame["Period"] = PERIOD_BY_SHEET[sheet_name]
    frame["SourceSheet"] = sheet_name
    frames.append(frame)

data = pd.concat(frames, ignore_index=True)
if len(data) != sum(len(frame) for frame in sheet_frames.values()):
    raise RuntimeError("El número de filas consolidadas no coincide con las hojas fuente.")

del frames, sheet_frames, excel, frame, sheet_name

print(f"Dimensiones consolidadas: {data.shape}")
print("Filas por fuente:")
display(data["SourceSheet"].value_counts().rename("Filas").to_frame())
print("Muestra consolidada:")
display(data.head(3))

Dimensiones consolidadas: (1067371, 10)
Filas por fuente:


,Filas
SourceSheet,
Year 2010-2011,541910
Year 2009-2010,525461


Muestra consolidada:


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,Period,SourceSheet
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom,2009-2010,Year 2009-2010
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,2009-2010,Year 2009-2010
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,2009-2010,Year 2009-2010


## 3. Reglas de calidad

| Dimensión | Regla o criterio operativo | Interpretación |
|---|---|---|
| Completitud | Contar nulos y textos vacíos por campo; foco en `Description` y `Customer ID`. | Un valor faltante reduce la información disponible, pero no invalida automáticamente la línea. |
| Unicidad | Contar repeticiones exactas en las ocho columnas originales y repeticiones de la clave candidata `Invoice` + `StockCode` + `InvoiceDate`. | La factura no es única por fila: contiene líneas de productos. Repeticiones requieren contexto antes de deduplicar. |
| Validez | Verificar identificador de factura con formato numérico o prefijos `C` (cancelación) o `A` (ajuste de deuda) seguidos de dígitos, códigos no vacíos, cantidades enteras, fechas válidas y precios finitos. | Un fallo de formato es una alerta; valores negativos/cero se contabilizan aparte y no se marcan por defecto como inválidos. |
| Consistencia | Verificar correspondencia `Period`–`SourceSheet`; revisar si una factura en una misma hoja tiene varias fechas/clientes/países y si un código de producto tiene descripciones diferentes. | Variaciones pueden ser errores, cambios históricos o casos legítimos; se investigan, no se corrigen por suposición. |
| Plausibilidad | Investigar facturas `C`, cantidades negativas, precios cero/negativos y extremos mediante cerca superior de Tukey (`Q3 + 1.5 × IQR`). | Son señales para revisión; las devoluciones, muestras, descuentos y pedidos grandes pueden ser legítimos. |

Los umbrales de Tukey son una herramienta exploratoria, no límites comerciales. No se asigna un puntaje global: no se han definido tolerancias de negocio para considerar aceptables estas alertas.

## 4. Completitud

Se cuantifican nulos y, en campos textuales, cadenas vacías o compuestas solo por espacios. Los porcentajes se calculan sobre el total de filas consolidadas.

In [4]:
TEXT_COLUMNS = ("Invoice", "StockCode", "Description", "Country")
missing_counts = {}
for column in EXPECTED_COLUMNS:
    missing = data[column].isna()
    if column in TEXT_COLUMNS:
        missing = missing | data[column].astype("string").str.strip().eq("").fillna(False)
    missing_counts[column] = int(missing.sum())

completeness = pd.DataFrame({
    "Nulos o vacíos": pd.Series(missing_counts),
})
completeness["Porcentaje de filas"] = (
    completeness["Nulos o vacíos"] / len(data) * 100
).round(3)
completeness.index.name = "Campo"
display(completeness.sort_values("Nulos o vacíos", ascending=False))

for column in ("Description", "Customer ID"):
    count = missing_counts[column]
    print(f"{column}: {count:,} filas ({count / len(data) * 100:.3f}%) sin valor")

,Nulos o vacíos,Porcentaje de filas
Campo,,
Customer ID,243007,22.767
Description,4382,0.411
StockCode,0,0.000
Invoice,0,0.000
Quantity,0,0.000
InvoiceDate,0,0.000
Price,0,0.000
Country,0,0.000


Description: 4,382 filas (0.411%) sin valor
Customer ID: 243,007 filas (22.767%) sin valor


## 5. Unicidad y duplicados

La clave de factura por sí sola no identifica una línea. Se miden duplicados exactos en los ocho campos fuente (copias adicionales y filas involucradas) y coincidencias de la clave candidata `Invoice` + `StockCode` + `InvoiceDate`. Estas coincidencias pueden ser líneas repetidas legítimas y no se eliminan.

In [5]:
exact_duplicate_mask = data.duplicated(subset=list(EXPECTED_COLUMNS), keep="first")
exact_duplicate_rows_mask = data.duplicated(subset=list(EXPECTED_COLUMNS), keep=False)
candidate_key = ["Invoice", "StockCode", "InvoiceDate"]
candidate_duplicate_rows = data.duplicated(subset=candidate_key, keep=False)
invoice_line_counts = data.groupby(["SourceSheet", "Invoice"], dropna=False, sort=False).size()

uniqueness = pd.DataFrame([
    {"Medida": "Facturas distintas dentro de su hoja", "Resultado": int(len(invoice_line_counts)),
     "Nota": "Una factura puede contener varias líneas; no se espera unicidad por fila."},
    {"Medida": "Copias exactas adicionales (8 campos fuente)", "Resultado": int(exact_duplicate_mask.sum()),
     "Nota": "Repeticiones después de la primera copia."},
    {"Medida": "Filas que participan en duplicados exactos", "Resultado": int(exact_duplicate_rows_mask.sum()),
     "Nota": "Incluye primera copia y repeticiones."},
    {"Medida": "Filas repetidas en clave candidata", "Resultado": int(candidate_duplicate_rows.sum()),
     "Nota": "Invoice + StockCode + InvoiceDate; no es una clave confirmada por la fuente."},
    {"Medida": "Máximo de líneas por factura", "Resultado": int(invoice_line_counts.max()),
     "Nota": "Calculado dentro de cada hoja fuente."},
])
display(uniqueness)
if exact_duplicate_rows_mask.any():
    print("Ejemplos de filas exactamente repetidas (sin eliminarlas):")
    display(data.loc[exact_duplicate_rows_mask, list(EXPECTED_COLUMNS) + ["SourceSheet"]].head(10))

,Medida,Resultado,Nota
0,Facturas distintas dentro de su hoja,54716,Una factura puede contener varias líneas; no s...
1,Copias exactas adicionales (8 campos fuente),34335,Repeticiones después de la primera copia.
2,Filas que participan en duplicados exactos,67242,Incluye primera copia y repeticiones.
3,Filas repetidas en clave candidata,88566,Invoice + StockCode + InvoiceDate; no es una c...
4,Máximo de líneas por factura,1114,Calculado dentro de cada hoja fuente.


Ejemplos de filas exactamente repetidas (sin eliminarlas):


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,SourceSheet
362,489517,21913,VINTAGE SEASIDE JIGSAW PUZZLES,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom,Year 2009-2010
363,489517,21912,VINTAGE SNAKES & LADDERS,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom,Year 2009-2010
365,489517,21821,GLITTER STAR GARLAND WITH BELLS,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom,Year 2009-2010
367,489517,22319,HAIRCLIPS FORTIES FABRIC ASSORTED,12,2009-12-01 11:34:00,0.65,16329.0,United Kingdom,Year 2009-2010
368,489517,22130,PARTY CONE CHRISTMAS DECORATION,6,2009-12-01 11:34:00,0.85,16329.0,United Kingdom,Year 2009-2010
371,489517,21912,VINTAGE SNAKES & LADDERS,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom,Year 2009-2010
379,489517,21491,SET OF THREE VINTAGE GIFT WRAPS,1,2009-12-01 11:34:00,1.95,16329.0,United Kingdom,Year 2009-2010
383,489517,22130,PARTY CONE CHRISTMAS DECORATION,6,2009-12-01 11:34:00,0.85,16329.0,United Kingdom,Year 2009-2010
384,489517,22319,HAIRCLIPS FORTIES FABRIC ASSORTED,12,2009-12-01 11:34:00,0.65,16329.0,United Kingdom,Year 2009-2010
385,489517,21913,VINTAGE SEASIDE JIGSAW PUZZLES,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom,Year 2009-2010


## 6. Validez de formatos y dominios

El formato de factura admite enteros y texto con prefijo `C` seguido de dígitos. Los códigos de producto se revisan como cadenas alfanuméricas; cantidades deben ser enteras, fecha debe estar informada y precio debe ser numérico finito. Las cantidades negativas y los precios no positivos se presentan en el análisis de plausibilidad para no confundirlos con errores sintácticos.

In [6]:
invoice_text = data["Invoice"].astype("string").str.upper()
is_adjustment = invoice_text.str.startswith("A", na=False)
invoice_valid = invoice_text.str.fullmatch(r"(?:C|A)?\d+", na=False)
stock_text = data["StockCode"].astype("string").str.strip()
stock_valid = stock_text.notna() & stock_text.ne("").fillna(False)
stock_alphanumeric = stock_text.str.fullmatch(r"[A-Za-z0-9]+", na=False)
nonstandard_stock_code = stock_valid & ~stock_alphanumeric
quantity_numeric = pd.to_numeric(data["Quantity"], errors="coerce")
quantity_integer = quantity_numeric.notna() & quantity_numeric.mod(1).eq(0)
price_numeric = pd.to_numeric(data["Price"], errors="coerce")
price_finite = pd.Series(np.isfinite(price_numeric.fillna(0)), index=data.index) & price_numeric.notna()
customer_numeric = pd.to_numeric(data["Customer ID"], errors="coerce")
customer_integer = customer_numeric.isna() | customer_numeric.mod(1).eq(0)
country_text = data["Country"].astype("string")
country_valid = country_text.notna() & country_text.str.strip().ne("")

date_is_datetime = pd.api.types.is_datetime64_any_dtype(data["InvoiceDate"].dtype)
validity = pd.DataFrame([
    ("Formato Invoice fuera de patrones observados", int((~invoice_valid).sum())),
    ("StockCode vacío", int((~stock_valid).sum())),
    ("StockCode con caracteres especiales (revisar)", int(nonstandard_stock_code.sum())),
    ("Quantity no entera o no numérica", int((~quantity_integer).sum())),
    ("Price no finito o no numérico", int((~price_finite).sum())),
    ("Customer ID no entero (entre los informados)", int((~customer_integer).sum())),
    ("Country vacío", int((~country_valid).sum())),
    ("InvoiceDate nula", int(data["InvoiceDate"].isna().sum())),
], columns=["Regla", "Filas no conformes"])
print(f"InvoiceDate interpretada como fecha/hora: {date_is_datetime}")
display(validity)

InvoiceDate interpretada como fecha/hora: True


,Regla,Filas no conformes
0,Formato Invoice fuera de patrones observados,0
1,StockCode vacío,0
2,StockCode con caracteres especiales (revisar),202
3,Quantity no entera o no numérica,0
4,Price no finito o no numérico,0
5,Customer ID no entero (entre los informados),0
6,Country vacío,0
7,InvoiceDate nula,0


In [7]:
invalid_invoice_rows = data.loc[~invoice_valid, ["Invoice", "SourceSheet"]]
nonstandard_stock_rows = data.loc[nonstandard_stock_code, ["StockCode", "Description", "SourceSheet"]]
print("Invoice identifiers outside the observed numeric/C/A patterns:")
display(invalid_invoice_rows["Invoice"].astype("string").value_counts().rename_axis("Invoice").to_frame("Rows"))
print("A-prefixed invoice adjustments (examples):")
display(data.loc[is_adjustment, ["Invoice", "StockCode", "Description", "Quantity", "Price", "SourceSheet"]].drop_duplicates().head(10))
print("StockCode values with spaces or special characters (review only, not automatic errors):")
display(nonstandard_stock_rows.drop_duplicates(subset=["StockCode"]).head(10))

Invoice identifiers outside the observed numeric/C/A patterns:


,Rows
Invoice,


A-prefixed invoice adjustments (examples):


,Invoice,StockCode,Description,Quantity,Price,SourceSheet
179403,A506401,B,Adjust bad debt,1,-53594.36,Year 2009-2010
276274,A516228,B,Adjust bad debt,1,-44031.79,Year 2009-2010
403472,A528059,B,Adjust bad debt,1,-38925.87,Year 2009-2010
825443,A563185,B,Adjust bad debt,1,11062.06,Year 2010-2011
825444,A563186,B,Adjust bad debt,1,-11062.06,Year 2010-2011
825445,A563187,B,Adjust bad debt,1,-11062.06,Year 2010-2011


StockCode values with spaces or special characters (review only, not automatic errors):


,StockCode,Description,SourceSheet
18410,BANK CHARGES,Bank Charges,Year 2009-2010
30620,gift_0001_80,NaN,Year 2009-2010
32048,gift_0001_20,Dotcomgiftshop Gift Voucher £20.00,Year 2009-2010
40904,gift_0001_10,Dotcomgiftshop Gift Voucher £10.00,Year 2009-2010
41263,gift_0001_50,Dotcomgiftshop Gift Voucher £50.00,Year 2009-2010
45212,gift_0001_30,Dotcomgiftshop Gift Voucher £30.00,Year 2009-2010
75137,gift_0001_40,NaN,Year 2009-2010
75138,gift_0001_60,NaN,Year 2009-2010
75139,gift_0001_70,NaN,Year 2009-2010
75141,gift_0001_90,NaN,Year 2009-2010


## 7. Cancelaciones, cantidades y precios

En esta fuente, las facturas cuyo identificador empieza por `C` se consideran cancelaciones/notas de crédito para revisión. Se compara ese indicador con cantidades negativas, y se cuantifican por separado cantidad cero, precio cero y precio negativo. Una señal no se convierte automáticamente en una exclusión del conjunto.

In [8]:
is_cancellation = invoice_text.str.startswith("C", na=False)
is_negative_quantity = data["Quantity"] < 0
is_zero_quantity = data["Quantity"] == 0
is_zero_price = data["Price"] == 0
is_negative_price = data["Price"] < 0

review_counts = pd.DataFrame([
    ("Líneas con factura prefijo C", int(is_cancellation.sum())),
    ("Facturas distintas con prefijo C", int(data.loc[is_cancellation, "Invoice"].nunique())),
    ("Líneas C con cantidad negativa", int((is_cancellation & is_negative_quantity).sum())),
    ("Líneas C con cantidad no negativa", int((is_cancellation & ~is_negative_quantity).sum())),
    ("Cantidad negativa", int(is_negative_quantity.sum())),
    ("Cantidad cero", int(is_zero_quantity.sum())),
    ("Precio cero", int(is_zero_price.sum())),
    ("Precio negativo", int(is_negative_price.sum())),
], columns=["Hallazgo", "Filas o facturas"])
display(review_counts)
print("Cantidad y precio por grupos relevantes:")
display(data.assign(
    TipoFactura=np.where(is_cancellation, "C: cancelación", "Sin prefijo C"),
    CantidadNegativa=is_negative_quantity,
).groupby(["TipoFactura", "CantidadNegativa"], dropna=False).size()
  .rename("Filas").to_frame())

,Hallazgo,Filas o facturas
0,Líneas con factura prefijo C,19494
1,Facturas distintas con prefijo C,8292
2,Líneas C con cantidad negativa,19493
3,Líneas C con cantidad no negativa,1
4,Cantidad negativa,22950
5,Cantidad cero,0
6,Precio cero,6202
7,Precio negativo,5


Cantidad y precio por grupos relevantes:


Filas
TipoFactura    CantidadNegativa         
C: cancelación False                   1
               True                19493
Sin prefijo C  False             1044420
               True                 3457

In [9]:
by_source = data.assign(
    Cancelacion=is_cancellation,
    CantidadNegativa=is_negative_quantity,
    PrecioCero=is_zero_price,
    PrecioNegativo=is_negative_price,
).groupby("SourceSheet", observed=True).agg(
    filas=("Invoice", "size"),
    facturas_cancelacion=("Cancelacion", "sum"),
    cantidades_negativas=("CantidadNegativa", "sum"),
    precios_cero=("PrecioCero", "sum"),
    precios_negativos=("PrecioNegativo", "sum"),
)
print("Distribuci?n de los hallazgos por hoja:")
display(by_source)
print("Factura C con cantidad no negativa:")
display(data.loc[is_cancellation & ~is_negative_quantity,
    ["Invoice", "StockCode", "Description", "Quantity", "Price", "InvoiceDate", "SourceSheet"]])
print("Ejemplos de l?neas con precio cero:")
display(data.loc[is_zero_price,
    ["Invoice", "StockCode", "Description", "Quantity", "Price", "InvoiceDate", "SourceSheet"]].head(10))
print("Todas las l?neas con precio negativo:")
display(data.loc[is_negative_price,
    ["Invoice", "StockCode", "Description", "Quantity", "Price", "InvoiceDate", "SourceSheet"]])

Distribuci?n de los hallazgos por hoja:

,filas,facturas_cancelacion,cantidades_negativas,precios_cero,precios_negativos
SourceSheet,,,,,
Year 2009-2010,525461,10206,12326,3687,3
Year 2010-2011,541910,9288,10624,2515,2


Factura C con cantidad no negativa:


,Invoice,StockCode,Description,Quantity,Price,InvoiceDate,SourceSheet
76799,C496350,M,Manual,1,373.57,2010-02-01 08:24:00,Year 2009-2010


Ejemplos de l?neas con precio cero:


,Invoice,StockCode,Description,Quantity,Price,InvoiceDate,SourceSheet
263,489464,21733,85123a mixed,-96,0.0,2009-12-01 10:52:00,Year 2009-2010
283,489463,71477,short,-240,0.0,2009-12-01 10:52:00,Year 2009-2010
284,489467,85123A,21733 mixed,-192,0.0,2009-12-01 10:53:00,Year 2009-2010
470,489521,21646,NaN,-50,0.0,2009-12-01 11:44:00,Year 2009-2010
3114,489655,20683,NaN,-44,0.0,2009-12-01 17:26:00,Year 2009-2010
3161,489659,21350,NaN,230,0.0,2009-12-01 17:39:00,Year 2009-2010
3162,489660,35956,lost,-1043,0.0,2009-12-01 17:43:00,Year 2009-2010
3168,489663,35605A,damages,-117,0.0,2009-12-01 18:02:00,Year 2009-2010
3731,489781,84292,NaN,17,0.0,2009-12-02 11:45:00,Year 2009-2010
4296,489806,18010,NaN,-770,0.0,2009-12-02 12:42:00,Year 2009-2010


Todas las l?neas con precio negativo:


,Invoice,StockCode,Description,Quantity,Price,InvoiceDate,SourceSheet
179403,A506401,B,Adjust bad debt,1,-53594.36,2010-04-29 13:36:00,Year 2009-2010
276274,A516228,B,Adjust bad debt,1,-44031.79,2010-07-19 11:24:00,Year 2009-2010
403472,A528059,B,Adjust bad debt,1,-38925.87,2010-10-20 12:04:00,Year 2009-2010
825444,A563186,B,Adjust bad debt,1,-11062.06,2011-08-12 14:51:00,Year 2010-2011
825445,A563187,B,Adjust bad debt,1,-11062.06,2011-08-12 14:52:00,Year 2010-2011


## 8. Consistencia entre campos

Se comprueba que las etiquetas de procedencia correspondan entre sí. También se busca más de una fecha, cliente informado o país por factura dentro de una misma hoja y descripciones distintas para el mismo código de producto. Las descripciones se comparan tanto en forma original como tras quitar espacios y normalizar mayúsculas.

In [10]:
expected_period = data["SourceSheet"].map(PERIOD_BY_SHEET)
period_mismatch = data["Period"].ne(expected_period)
invoice_consistency = data.groupby(["SourceSheet", "Invoice"], dropna=False, sort=False).agg(
    fechas_distintas=("InvoiceDate", "nunique"),
    clientes_informados=("Customer ID", "nunique"),
    paises_distintos=("Country", "nunique"),
    lineas=("Invoice", "size"),
)
consistency = pd.DataFrame([
    ("Period no corresponde a SourceSheet", int(period_mismatch.sum())),
    ("Facturas con más de una fecha en la hoja", int(invoice_consistency["fechas_distintas"].gt(1).sum())),
    ("Facturas con más de un Customer ID informado", int(invoice_consistency["clientes_informados"].gt(1).sum())),
    ("Facturas con más de un país", int(invoice_consistency["paises_distintos"].gt(1).sum())),
], columns=["Regla de consistencia", "Facturas/filas señaladas"])
display(consistency)

normalized_description = data["Description"].astype("string").str.strip().str.upper()
description_audit = pd.DataFrame({
    "StockCode": data["StockCode"],
    "Descripción original": data["Description"],
    "Descripción normalizada": normalized_description,
}).groupby("StockCode", dropna=False).agg(
    descripciones_originales=("Descripción original", "nunique"),
    descripciones_normalizadas=("Descripción normalizada", "nunique"),
    filas=("Descripción original", "size"),
)
raw_variant_codes = int(description_audit["descripciones_originales"].gt(1).sum())
normalized_variant_codes = int(description_audit["descripciones_normalizadas"].gt(1).sum())
print(f"StockCode con varias descripciones originales: {raw_variant_codes:,}")
print(f"StockCode con varias descripciones tras normalizar espacios/mayúsculas: {normalized_variant_codes:,}")
print("Ejemplos de códigos con variantes normalizadas:")
display(description_audit.loc[description_audit["descripciones_normalizadas"].gt(1)]
        .sort_values("filas", ascending=False).head(10))

print("Rango temporal por hoja (la etiqueta de periodo procede de la fuente):")
display(data.groupby(["SourceSheet", "Period"], observed=True)["InvoiceDate"].agg(["min", "max", "count"]))

,Regla de consistencia,Facturas/filas señaladas
0,Period no corresponde a SourceSheet,0
1,Facturas con más de una fecha en la hoja,84
2,Facturas con más de un Customer ID informado,0
3,Facturas con más de un país,0


StockCode con varias descripciones originales: 1,232
StockCode con varias descripciones tras normalizar espacios/mayúsculas: 1,212
Ejemplos de códigos con variantes normalizadas:


,descripciones_originales,descripciones_normalizadas,filas
StockCode,,,
85123A,5,5,5829
22423,7,7,4424
85099B,4,4,4216
21212,2,2,3318
20725,2,2,3259
84879,3,3,2960
47566,2,2,2768
21232,2,2,2747
22197,3,3,2549


Rango temporal por hoja (la etiqueta de periodo procede de la fuente):


,,min,max,count
SourceSheet,Period,,,
Year 2009-2010,2009-2010,2009-12-01 07:45:00,2010-12-09 20:01:00,525461
Year 2010-2011,2010-2011,2010-12-01 08:26:00,2011-12-09 12:50:00,541910


## 9. Plausibilidad y valores atípicos

Se calculan cercas superiores de Tukey para **precios positivos** y para la **magnitud absoluta de Quantity**. Se reportan también extremos, sin asumir que una observación fuera de la cerca sea errónea. Los valores negativos se evalúan aparte, ya que pueden representar devoluciones o ajustes.

In [11]:
positive_prices = data.loc[data["Price"] > 0, "Price"]
price_q1, price_q3 = positive_prices.quantile([0.25, 0.75])
price_iqr = price_q3 - price_q1
price_upper_fence = price_q3 + 1.5 * price_iqr
price_outlier = data["Price"] > price_upper_fence

absolute_quantity = data["Quantity"].abs()
quantity_q1, quantity_q3 = absolute_quantity.quantile([0.25, 0.75])
quantity_iqr = quantity_q3 - quantity_q1
quantity_upper_fence = quantity_q3 + 1.5 * quantity_iqr
quantity_outlier = absolute_quantity > quantity_upper_fence

outlier_summary = pd.DataFrame([
    ("Price > 0", float(price_q1), float(price_q3), float(price_upper_fence), int(price_outlier.sum())),
    ("abs(Quantity)", float(quantity_q1), float(quantity_q3), float(quantity_upper_fence), int(quantity_outlier.sum())),
], columns=["Variable", "Q1", "Q3", "Cerca superior (Q3 + 1.5×IQR)", "Filas sobre la cerca"])
display(outlier_summary)
print("Máximos y mínimos observados:")
display(data[["Quantity", "Price"]].agg(["min", "max"]).T)

outlier_examples = data.loc[price_outlier | quantity_outlier,
    ["Invoice", "StockCode", "Description", "Quantity", "Price", "InvoiceDate", "Country", "Period", "SourceSheet"]].copy()
outlier_examples["LineAmount (Quantity × Price)"] = outlier_examples["Quantity"] * outlier_examples["Price"]
print("Ejemplos de líneas señaladas (no eliminadas):")
display(outlier_examples.assign(
    MagnitudCantidad=outlier_examples["Quantity"].abs(),
    PrecioSobreCerca=outlier_examples["Price"] > price_upper_fence,
).sort_values(["PrecioSobreCerca", "MagnitudCantidad"], ascending=False).head(10))

,Variable,Q1,Q3,Cerca superior (Q3 + 1.5×IQR),Filas sobre la cerca
0,Price > 0,1.25,4.15,8.5,68100
1,abs(Quantity),1.00,10.00,23.5,115571


Máximos y mínimos observados:


,min,max
Quantity,-80995.00,80995.0
Price,-53594.36,38970.0


Ejemplos de líneas señaladas (no eliminadas):


,Invoice,StockCode,Description,Quantity,Price,InvoiceDate,Country,Period,SourceSheet,LineAmount (Quantity × Price),MagnitudCantidad,PrecioSobreCerca
358819,524181,21621,VINTAGE UNION JACK BUNTING,408,8.87,2010-09-27 16:59:00,United Kingdom,2009-2010,Year 2009-2010,3618.96,408,True
420628,529613,22423,REGENCY CAKESTAND 3 TIER,360,10.95,2010-10-29 10:46:00,United Kingdom,2009-2010,Year 2009-2010,3942.00,360,True
936690,572209,23556,LANDMARK FRAME COVENT GARDEN,300,10.95,2011-10-21 12:08:00,United Kingdom,2010-2011,Year 2010-2011,3285.00,300,True
936691,572209,23554,LANDMARK FRAME OXFORD STREET,300,10.95,2011-10-21 12:08:00,United Kingdom,2010-2011,Year 2010-2011,3285.00,300,True
936692,572209,23555,LANDMARK FRAME NOTTING HILL,300,10.95,2011-10-21 12:08:00,United Kingdom,2010-2011,Year 2010-2011,3285.00,300,True
936693,572209,23558,LANDMARK FRAME LONDON BRIDGE,300,10.95,2011-10-21 12:08:00,United Kingdom,2010-2011,Year 2010-2011,3285.00,300,True
936695,572209,23553,LANDMARK FRAME CAMDEN TOWN,300,10.95,2011-10-21 12:08:00,United Kingdom,2010-2011,Year 2010-2011,3285.00,300,True
936696,572209,23557,LANDMARK FRAME BAKER STREET,300,10.95,2011-10-21 12:08:00,United Kingdom,2010-2011,Year 2010-2011,3285.00,300,True
548724,538191,22423,REGENCY CAKESTAND 3 TIER,272,10.95,2010-12-10 10:49:00,United Kingdom,2010-2011,Year 2010-2011,2978.40,272,True
351435,523576,22423,REGENCY CAKESTAND 3 TIER,256,10.95,2010-09-22 16:26:00,United Kingdom,2009-2010,Year 2009-2010,2803.20,256,True


## 10. Resumen cuantificado y tratamiento recomendado

Las métricas siguientes resumen los hallazgos anteriores. Los porcentajes se calculan sobre las filas consolidadas, excepto completitud, que también informa celdas faltantes. El tratamiento mantiene el archivo de origen íntegro.

In [12]:
n_rows = len(data)
exact_duplicate_excess = int(exact_duplicate_mask.sum())
missing_cell_count = int(sum(missing_counts.values()))
summary = pd.DataFrame([
    ("Completitud", f"{missing_cell_count:,} celdas faltantes/vacías de {n_rows * len(EXPECTED_COLUMNS):,}",
     "Description y Customer ID concentran los faltantes analizados."),
    ("Unicidad", f"{exact_duplicate_excess:,} copias exactas adicionales; {int(exact_duplicate_rows_mask.sum()):,} filas involucradas",
     "Invoice no es una clave de línea; validar duplicados contra documentos fuente."),
    ("Validez", f"{int((~invoice_valid).sum()):,} facturas fuera de patrones; {int((~stock_valid).sum()):,} codigos vacios; {int(nonstandard_stock_code.sum()):,} formatos especiales revisables", "C y A tienen significado de negocio; codigos especiales pueden ser cargos o vales."),
    ("Consistencia", f"{int(period_mismatch.sum()):,} etiquetas Period/SourceSheet discordantes; {int(invoice_consistency['fechas_distintas'].gt(1).sum()):,} facturas con varias fechas",
     "Una discordancia comprobable requiere corregir metadatos desde la hoja fuente."),
    ("Plausibilidad", f"{int(is_negative_quantity.sum()):,} cantidades negativas; {int(is_zero_price.sum()):,} precios cero; {int(is_negative_price.sum()):,} precios negativos",
     "Separar y revisar devoluciones, artículos gratuitos y ajustes; no tratarlos como ventas ordinarias."),
    ("Atípicos", f"{int(price_outlier.sum()):,} precios y {int(quantity_outlier.sum()):,} magnitudes de cantidad sobre su cerca IQR",
     "IQR es una alerta exploratoria; verificar pedidos grandes, artículos especiales y correcciones."),
], columns=["Dimensión", "Hallazgo cuantificado", "Lectura inicial"])
display(summary)

treatments = pd.DataFrame([
    ("Description faltante", f"{missing_counts['Description']:,} filas", "Conservar; no inferir descripciones. Marcar como desconocida en análisis por producto; limita la lectura del catálogo, pero no invalida los otros campos."),
    ("Customer ID faltante", f"{missing_counts['Customer ID']:,} filas", "Conservar; no imputar identidad. Reportar cobertura y excluir solo de métricas que exijan cliente, documentando la exclusión analítica."),
    ("StockCode con caracteres especiales", f"{int(nonstandard_stock_code.sum()):,} filas", "Conservar: ejemplos como BANK CHARGES y codigos de vales son categorias legitimas, no errores de formato por si solos."),
    ("Factura con prefijo A", f"{int(is_adjustment.sum()):,} filas", "Conservar y segmentar como ajuste de deuda; revisar junto con StockCode B y Description Adjust bad debt, sin tratarlo como venta ordinaria."),
    ("Factura con prefijo C", f"{int(is_cancellation.sum()):,} líneas", "Conservar y segmentar como cancelación/nota de crédito. No mezclar silenciosamente con ventas brutas; confirmar regla contable para ventas netas."),
    ("Quantity negativa", f"{int(is_negative_quantity.sum()):,} filas", "Conservar y marcar como posible devolución/ajuste. Investigar junto con factura y fecha; excluir solo si la métrica define explícitamente ventas brutas."),
    ("Price igual a cero o negativo", f"{int((is_zero_price | is_negative_price).sum()):,} filas", "Conservar y revisar. Cero puede ser muestra/promocion; los negativos observados corresponden a ajustes A de deuda; confirmar regla contable."),
    ("Duplicado exacto", f"{exact_duplicate_excess:,} copias adicionales", "No borrar automáticamente: una repetición puede ser una línea legítima. Confirmar con identificador de línea/documento y medir impacto antes de deduplicar."),
    ("Valor atípico por IQR", f"{int((price_outlier | quantity_outlier).sum()):,} filas en al menos una regla", "Conservar y señalar para revisión. El umbral estadístico no prueba error; eliminar podría descartar pedidos reales o ajustes."),
], columns=["Caso", "Volumen", "Tratamiento y justificación"])
display(treatments)

print(
    f"Conclusión: se revisaron {n_rows:,} líneas de las dos hojas. "
    "No se eliminó ni imputó ninguna fila; las banderas y umbrales sirven para investigar, "
    "y cualquier tratamiento posterior debe documentar su alcance y efecto en los resultados."
)

,Dimensión,Hallazgo cuantificado,Lectura inicial
0,Completitud,"247,389 celdas faltantes/vacías de 8,538,968",Description y Customer ID concentran los falta...
1,Unicidad,"34,335 copias exactas adicionales; 67,242 fila...",Invoice no es una clave de línea; validar dupl...
2,Validez,0 facturas fuera de patrones; 0 codigos vacios...,C y A tienen significado de negocio; codigos e...
3,Consistencia,0 etiquetas Period/SourceSheet discordantes; 8...,Una discordancia comprobable requiere corregir...
4,Plausibilidad,"22,950 cantidades negativas; 6,202 precios cer...","Separar y revisar devoluciones, artículos grat..."
5,Atípicos,"68,100 precios y 115,571 magnitudes de cantida...",IQR es una alerta exploratoria; verificar pedi...


,Caso,Volumen,Tratamiento y justificación
0,Description faltante,"4,382 filas",Conservar; no inferir descripciones. Marcar co...
1,Customer ID faltante,"243,007 filas",Conservar; no imputar identidad. Reportar cobe...
2,StockCode con caracteres especiales,202 filas,Conservar: ejemplos como BANK CHARGES y codigo...
3,Factura con prefijo A,6 filas,Conservar y segmentar como ajuste de deuda; re...
4,Factura con prefijo C,"19,494 líneas",Conservar y segmentar como cancelación/nota de...
5,Quantity negativa,"22,950 filas",Conservar y marcar como posible devolución/aju...
6,Price igual a cero o negativo,"6,207 filas",Conservar y revisar. Cero puede ser muestra/pr...
7,Duplicado exacto,"34,335 copias adicionales",No borrar automáticamente: una repetición pued...
8,Valor atípico por IQR,"183,004 filas en al menos una regla",Conservar y señalar para revisión. El umbral e...


Conclusión: se revisaron 1,067,371 líneas de las dos hojas. No se eliminó ni imputó ninguna fila; las banderas y umbrales sirven para investigar, y cualquier tratamiento posterior debe documentar su alcance y efecto en los resultados.
